## 使用pytorch对Fashion-MNIST进行预测
#### 1.导入需要引用的包


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

### 2.下载数据集
    PyTorch 提供了两个用于处理数据的基本函数：torch.utils.data.DataLoader 和 torch.utils.data.Dataset。
    Dataset 存储样本及其对应的标签，像一本“菜谱”——告诉你有哪些食材（样本）以及每道菜怎么做（如何加载/预处理）；
    DataLoader 则将 Dataset 封装成一个可迭代对象，像一个“厨房流水线”——按需批量取菜、洗菜、切菜（batching, shuffling, multiprocessing）。

In [ ]:
# 从公开数据集下载训练数据
training_data = datasets.FashionMNIST(
    root="../data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# 从公开数据集下载测试数据
test_data = datasets.FashionMNIST(
    root="../data",
    train=False,
    download=True,
    transform=ToTensor(),
)

### 3.数据处理
    我们将数据集作为参数传递给 DataLoader。DataLoader 封装了一个可迭代对象，
    用于存储我们的数据集，并支持自动批处理、采样、打乱顺序和多进程数据加载。
    这里我们定义了批次大小为 64，也就是说，DataLoader 可迭代对象中的每个元素都会返回一个包含 64 个特征和标签的批次。

In [ ]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

### 4.选择正确的设备处理模型

In [ ]:
# 正确的设备选择
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
    else "cpu"
)

print(f"使用 {device} 设备")

### 5.对模型进行定义
🔹 整体目标
这段代码的目标是：
    定义一个简单的全连接神经网络（MLP）。
    使用 nn.Module 作为基类来封装模型结构。
    将模型部署到可用的硬件加速器（如 GPU/CUDA、MPS 等）上。
    实现前向传播过程（forward pass）。

In [ ]:
# Define model
class NeuralNetwork(nn.Module):     #  继承 nn.Module，所有 PyTorch 的神经网络都应继承自 nn.Module。 这个基类提供了参数管理、前向传播、保存/加载模型等功能。
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()     #将输入张量从形状 (batch_size, 28, 28) 压平为 (batch_size, 784)。因为后续的 Linear 层只能处理一维向量（除了第一个维度是 batch）。
        self.linear_relu_stack = nn.Sequential(         #  是一种顺序容器，会按顺序执行列表中的模块。每一层依次作用于前一层输出。
            nn.Linear(28*28, 512),      # 输入层：784 → 512，全连接层
            nn.ReLU(),      # 激活函数：引入非线性
            nn.Linear(512, 512),        # 隐藏层：512 → 512
            nn.ReLU(),      # 激活函数
            nn.Linear(512, 10)      # 输出层：分类任务，10 类（如 MNIST 数字 0~9，衣服类型等等）
        )

    def forward(self, x):       # 前向传播：forward 函数
        x = self.flatten(x)     # 输入 x 先被压平（flatten），然后送入 linear_relu_stack
        logits = self.linear_relu_stack(x)
        return logits       # 返回的是原始输出（logits），不是概率，因为没有做softmax

model = NeuralNetwork().to(device)      # 创建模型实例并移动到设备
print(model)

### 6.定义损失函数和优化器

In [ ]:
loss_fn = nn.CrossEntropyLoss()     # nn.CrossEntropyLoss()这是一个分类任务中最常用的损失函数。它内部自动对模型输出（logits）做 softmax，然后计算 负对数似然损失（NLL Loss）。
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)   # 使用随机梯度下降（Stochastic Gradient Descent） 优化器。model.parameters()：告诉优化器“哪些参数需要更新”——即模型中所有可学习的权重和偏置。lr=1e-3：学习率（learning rate），控制每次参数更新的步长。

### 7.训练函数
🔹 整体目标
这段代码的目标是：
    定义一个训练函数 train()；
    在每个 batch 上执行前向传播 → 计算损失 → 反向传播 → 更新参数；
    使用交叉熵损失和随机梯度下降（SGD）优化器；
    打印训练过程中的 loss 以监控进度。


In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    """
    :param dataloader: 提供批量数据（如 DataLoader 实例）
    :param model: 要训练的神经网络；
    :param loss_fn: 损失函数
    :param optimizer: 优化器。
    :return:
    """
    size = len(dataloader.dataset)      # 获取整个训练集样本总数（用于打印进度）。
    model.train()       # 非常重要！ 将模型切换到 训练模式（training mode）。 影响某些层的行为，例如： Dropout：训练时启用，推理时关闭； BatchNorm：训练时用 batch 统计量，推理时用全局统计量。 如果不调用 .train()，模型可能表现异常（尤其用了这些层时）。
    for batch, (X, y) in enumerate(dataloader):   #  每次返回一个 batch 的 (X, y)，X：输入图像。y：标签
        X, y = X.to(device), y.to(device)

        pred = model(X)     # 调用模型的 forward() 方法，得到 logits（shape [64, 10]），向前传播
        loss = loss_fn(pred, y)     # 计算当前 batch 的平均损失（标量）。

        # 反向传播更新参数
        loss.backward()     # 自动计算损失对所有可学习参数的梯度（通过反向传播算法）。 梯度会累加到每个参数的 .grad 属性中。
        optimizer.step()        # 根据当前梯度（.grad）和优化器规则（如 SGD: w = w - lr * grad）更新参数。
        optimizer.zero_grad()       #  清空梯度缓冲区（将所有 .grad 设为 None 或 0）。 ⚠️ 如果不清零，梯度会累积！ 下一次 backward() 会把新梯度加到旧梯度上，导致错误更新。
        if batch % 100 == 0:        # 打印训练进度
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

### 8.测试函数
🔹 整体目标
这段 test() 函数的目标是：
    在测试集上评估训练好的模型；
    计算 平均损失（Avg loss） 和 分类准确率（Accuracy）；
    不进行反向传播或参数更新（因为只是评估，不是训练）；
    使用 torch.no_grad() 提升效率并节省内存。

In [ ]:
def test(dataloader, model, loss_fn):
    """
    在测试集上评估训练好的模型； 计算 平均损失（Avg loss） 和 分类准确率（Accuracy）； 不进行反向传播或参数更新（因为只是评估，不是训练）； 使用 torch.no_grad() 提升效率并节省内存。
    :param dataloader:  测试数据加载器（如 DataLoader(test_dataset)）；
    :param model:   已训练的模型；
    :param loss_fn: 与训练时相同的损失函数（如 CrossEntropyLoss）。
    """
    size = len(dataloader.dataset)      # 总样本数，如 10000（MNIST 测试集）
    num_batches = len(dataloader)       # 总 batch 数，如 157（10000 / 64 ≈ 157）,这两个值用于后续计算平均损失和整体准确率。
    model.eval()        # 将模型设置为 评估模式（evaluation mode）。对应训练时的 model.train()，两者必须成对使用！
    test_loss, correct = 0, 0       # 累计所有 batch 的损失总和；    累计预测正确的样本总数。
    with torch.no_grad():       # with类似与java中的try catch finally，禁用梯度计算：with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)     # 前向传播，得到预测 logits
            test_loss += loss_fn(pred, y).item()        # loss_fn(pred, y)：计算当前 batch 的平均损失（标量张量）；
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()  # pred.argmax(1)在对所有类别预测中找到数值最大的--预测最像的，如果和标签y相等证明预测对了
    test_loss /= num_batches        #   平均每个 batch 的损失（注意：不是每个样本！但因为 CrossEntropyLoss 默认对 batch 求平均，所以等价于平均每个样本的 loss）；
    correct /= size     # 整体准确率（正确样本数 / 总样本数）。
    print(f"失败: \n 精确度: {(100*correct):>0.1f}%, 平均损失: {test_loss:>8f} \n")

epochs = 10
for t in range(epochs):     # 类似于java的 for(int i = 0;i<10;i++)
    print(f"回合 {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("完成!")

### 7.保存模型

In [ ]:
torch.save(model.state_dict(), "../data/fashion-MNIST/model.pth")
print("Saved PyTorch Model State to model.pth")

### 8.加载模型

In [ ]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("../data/fashion-MNIST/model.pth", weights_only=True))

### 9.使用模型

In [ ]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'预测: "{predicted}", 实际: "{actual}"')